# World Model Training Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SharathSPhD/dreamprice/blob/master/notebooks/world_model_training.ipynb)

End-to-end demonstration of the DreamPrice training pipeline:

1. Build the `DominicksSequenceDataset` from raw data
2. Construct the `MambaWorldModel` with RSSM + Mamba-2 backbone
3. Run the DreamerV3 three-phase training loop
4. Inspect loss curves and latent representations

This notebook runs a short training demonstration (10 steps) on synthetic
data. For full training on Dominick's data, use `scripts/train.py` inside
the Docker container with GPU access.

In [ ]:
# Install DreamPrice (skip if already installed)
import subprocess, sys
try:
    import retail_world_model
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'torch', '--index-url', 'https://download.pytorch.org/whl/cpu'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '..'])

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

## 1. Model Architecture

The world model consists of:
- **Encoder**: Maps observations to the posterior stochastic state z_t
- **Mamba-2 Backbone**: Deterministic recurrence via selective state-space model
- **Prior MLP**: Predicts the next stochastic state from h_t
- **Observation Decoder**: Reconstructs x_t from (h_t, z_t)
- **Reward Ensemble**: 5 independent heads for MOPO-style LCB pessimism
- **Continue Head**: Predicts episode continuation probability

In [ ]:
from retail_world_model.models.world_model import MambaWorldModel

obs_dim = 32
act_dim = 4
d_model = 64
n_cat, n_cls = 8, 8

model = MambaWorldModel(
    obs_dim=obs_dim,
    act_dim=act_dim,
    d_model=d_model,
    n_cat=n_cat,
    n_cls=n_cls,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {n_params:,}')
print(f'Trainable parameters: {n_trainable:,}')
print(f'Model: {type(model).__name__}')

In [ ]:
# Forward pass with random data
B, T = 2, 10
x = torch.randn(B, T, obs_dim, device=device)
a = torch.randn(B, T, act_dim, device=device)

output = model.forward(x, a)
print('Forward pass output keys:')
for key, val in output.items():
    if isinstance(val, torch.Tensor):
        print(f'  {key}: {val.shape}')
    else:
        print(f'  {key}: {type(val).__name__}')

## 2. ELBO Loss

The world model is trained with a modified ELBO loss:

$$\mathcal{L} = \beta_{\text{pred}} \cdot (\mathcal{L}_{\text{recon}} + \mathcal{L}_{\text{reward}} + \mathcal{L}_{\text{continue}}) + \mathcal{L}_{\text{KL}}$$

- **Reconstruction**: Symlog MSE between predicted and actual observations
- **Reward**: Twohot cross-entropy for distributional reward prediction
- **Continue**: Binary cross-entropy for episode continuation
- **KL**: Balanced KL divergence between posterior and prior (DreamerV3-style)

In [ ]:
from retail_world_model.training.losses import elbo_loss

batch = {
    'x_BT': torch.randn(B, T, obs_dim, device=device),
    'a_BT': torch.randn(B, T, act_dim, device=device),
    'r_BT': torch.randn(B, T, device=device),
    'done_BT': torch.zeros(B, T, device=device),
}

losses = elbo_loss(batch, model)
print('ELBO loss components:')
for key, val in losses.items():
    print(f'  {key}: {val.item():.4f}')

## 3. Training Loop (Short Demo)

The DreamerV3 three-phase loop:
- **Phase A**: World model update (ELBO minimization)
- **Phase B**: Imagination rollout with the actor to generate synthetic trajectories
- **Phase C**: Actor-critic update on imagined trajectories (REINFORCE + distributional critic)

In [ ]:
from retail_world_model.applications.pricing_policy import ActorCritic
from retail_world_model.training.trainer import DreamerTrainer

state_dim = d_model + n_cat * n_cls
ac = ActorCritic(
    state_dim=state_dim,
    n_skus=act_dim,
    action_dim=21,
).to(device)

dummy_dataset = torch.utils.data.TensorDataset(torch.randn(10, T, obs_dim))
trainer = DreamerTrainer(
    model=model,
    actor_critic=ac,
    dataset=dummy_dataset,
)

# Run 10 training steps and track losses
loss_history = []
for step in range(10):
    batch = {
        'x_BT': torch.randn(B, T, obs_dim, device=device),
        'a_BT': torch.randn(B, T, act_dim, device=device),
        'r_BT': torch.randn(B, T, device=device),
        'done_BT': torch.zeros(B, T, device=device),
    }
    metrics = trainer.train_step(batch)
    loss_history.append(metrics.get('wm/total', 0.0))
    if step % 2 == 0:
        print(f'Step {step}: wm_loss={metrics.get("wm/total", 0):.4f}')

print(f'\nFinal WM loss: {loss_history[-1]:.4f}')

In [ ]:
# Plot loss curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_history, 'o-', color='steelblue', linewidth=2)
ax.set_xlabel('Training Step')
ax.set_ylabel('World Model Loss')
ax.set_title('World Model Training Loss (Demo)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Imagination Rollout

After training, the world model can imagine future trajectories in latent
space without needing real observations. This is the basis of the
DreamerV3 actor-critic training.

In [ ]:
# Demonstrate imagination rollout
model.train(False)
state = model.reset_state(batch_size=1)
z_t = state['z'].to(device)

imagined_rewards = []
imagined_continues = []

for t in range(13):
    a_t = torch.randn(1, act_dim, device=device)
    with torch.no_grad():
        result = model.imagine_step(z_t, a_t)
    z_t = result['z']
    imagined_rewards.append(result['r_mean'].item())
    imagined_continues.append(result['continue'].item())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(13), imagined_rewards, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Predicted Reward')
axes[0].set_title('Imagined Reward Trajectory')
axes[0].grid(True, alpha=0.3)

axes[1].plot(imagined_continues, 'o-', color='#55A868', linewidth=2)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Continue Probability')
axes[1].set_title('Imagined Continue Probability')
axes[1].set_ylim(0, 1.1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
model.train(True)

## 5. Loading the Trained 100K Checkpoint

The full DreamPrice model was trained for 100K gradient steps on the Dominick's
canned soup category using an NVIDIA DGX Spark (GB10 GPU). Training completed in
approximately 2.6 hours. The trained checkpoint and training metrics are available
on HuggingFace:

- **Model**: [qbz506/dreamprice-cso](https://huggingface.co/qbz506/dreamprice-cso)
- **Dataset**: [qbz506/dreamprice-dominicks-cso](https://huggingface.co/datasets/qbz506/dreamprice-dominicks-cso)
- **Demo**: [qbz506/dreamprice-demo](https://huggingface.co/spaces/qbz506/dreamprice-demo)

### Final Training Metrics (100K steps)

| Metric | Value |
|--------|-------|
| World Model ELBO | 22.44 |
| Reconstruction Loss | 0.001 |
| KL Divergence | 19.20 |
| Reward Prediction | 3.16 |
| Actor Return (mean) | 124.33 |
| Critic Loss | 2.50 |
| Training Time | ~2.6 hours |

### Full Training Pipeline

```bash
# Inside Docker container (make shell)
python scripts/estimate_elasticity.py --category cso
python scripts/train.py n_steps=100000
python scripts/evaluate_world_model.py --checkpoint checkpoints/step_0100000.pt
```

See the [GitHub repository](https://github.com/SharathSPhD/dreamprice) for
Docker setup and full training pipeline details.